# Laboratorio de Agentes de IA — Unidad 2.3

Material de apoyo practico para la Unidad 2.3 (Agentes de Inteligencia Artificial) del
curso Estrategias de Transformacion Digital.

Nota importante sobre este notebook: en una implementacion real, el "cerebro" de un
agente es un modelo de lenguaje grande (LLM) al que se accede mediante una API externa
(por ejemplo, la API de OpenAI, Anthropic o similar), lo cual requiere una llave de API
y tiene un costo por uso. Para que este notebook se pueda ejecutar completo sin
necesidad de esa llave ni de conexion a un servicio externo, se construye una funcion
`modelo_lenguaje_simulado` que imita, con reglas simples, el tipo de decisiones que
tomaria un LLM real. Esta simplificacion permite mostrar la mecanica completa de
prompting, tool calling, agentes y sistemas multiagentes; en un caso real, esa funcion
simulada se reemplazaria por una llamada real a la API del modelo que la organizacion
decida usar (build vs. buy vs. adopt, otra vez).

Contenido:

1. Prompting: construccion de prompts estructurados
2. Un modelo de lenguaje simulado, para no depender de una API externa
3. Agente basico: el ciclo observar, decidir, actuar
4. Tool calling: un agente que reprograma citas medicas
5. Sistema multiagente: un reporte de inteligencia de mercado
6. Simulacion de un flujo tipo n8n
7. Riesgos en la practica: alucinaciones y confirmacion humana


## 1. Prompting: construccion de prompts estructurados

Antes de automatizar nada, conviene separar la construccion del prompt de su uso, para
poder reutilizar la misma estructura con distintos datos. Aqui se construye una funcion
que ensambla un prompt a partir de sus partes (rol, instruccion, formato esperado y
ejemplos), siguiendo la estructura vista en el texto guia de la Unidad 2.3.

In [1]:
def construir_prompt(rol, instruccion, formato_esperado, ejemplos=None, entrada=""):
    """Ensambla un prompt estructurado a partir de sus componentes.

    rol: el papel que debe asumir el modelo (texto libre)
    instruccion: la tarea concreta que se le pide
    formato_esperado: como debe verse la respuesta
    ejemplos: lista opcional de pares (entrada, salida) para few-shot
    entrada: el texto concreto sobre el que debe actuar el modelo
    """
    partes = [f"Rol: {rol}"]  # primera parte del prompt: el rol asignado al modelo
    partes.append(f"Instruccion: {instruccion}")  # segunda parte: la tarea a realizar
    partes.append(f"Formato de salida esperado: {formato_esperado}")  # tercera parte: el formato de la respuesta

    if ejemplos:  # si se pasaron ejemplos, se agregan al prompt (tecnica few-shot)
        partes.append("Ejemplos:")  # encabezado de la seccion de ejemplos
        for entrada_ejemplo, salida_ejemplo in ejemplos:  # itera sobre cada par (entrada, salida) de ejemplo
            partes.append(f"  Entrada: {entrada_ejemplo} -> Salida: {salida_ejemplo}")  # agrega el ejemplo formateado

    partes.append(f"Entrada actual: {entrada}")  # ultima parte: el texto real que el modelo debe procesar
    return "\n".join(partes)  # une todas las partes en un solo texto, separadas por saltos de linea


# Ejemplo: el prompt de clasificacion de quejas visto en el texto guia de la Unidad 2.3
prompt_ejemplo = construir_prompt(
    rol="agente de atencion al cliente de una empresa de telecomunicaciones",
    instruccion="clasifica la queja del cliente en una sola categoria de la lista",
    formato_esperado="una sola palabra: facturacion, calidad_servicio, atencion_cliente u otro",
    ejemplos=[
        ("me cobraron el doble este mes", "facturacion"),
        ("la senal se cae constantemente", "calidad_servicio"),
    ],
    entrada="llevo tres dias esperando que alguien me devuelva la llamada",
)

print(prompt_ejemplo)  # muestra el prompt completo ya ensamblado


Rol: agente de atencion al cliente de una empresa de telecomunicaciones
Instruccion: clasifica la queja del cliente en una sola categoria de la lista
Formato de salida esperado: una sola palabra: facturacion, calidad_servicio, atencion_cliente u otro
Ejemplos:
  Entrada: me cobraron el doble este mes -> Salida: facturacion
  Entrada: la senal se cae constantemente -> Salida: calidad_servicio
Entrada actual: llevo tres dias esperando que alguien me devuelva la llamada


## 2. Un modelo de lenguaje simulado

Esta funcion reemplaza, solo para efectos de este notebook, la llamada a un LLM real.
Internamente usa reglas simples de coincidencia de palabras clave para decidir que
responder, segun el tipo de prompt que recibe. No es inteligencia artificial real: es un
simulador que permite construir y probar la logica de un agente sin depender de una API
externa.

In [2]:
def modelo_lenguaje_simulado(prompt, contexto=None):
    """Simula la respuesta de un modelo de lenguaje a partir de un prompt.

    contexto: diccionario opcional con informacion adicional que el "modelo" puede usar
    para simular que tiene memoria o acceso a resultados de herramientas previas.
    """
    texto = prompt.lower()  # se trabaja en minusculas para que la busqueda de palabras clave sea mas simple
    contexto = contexto or {}  # si no se pasa contexto, se usa un diccionario vacio

    # --- Simulacion de clasificacion de quejas (seccion 1) ---
    if "clasifica la queja" in texto:  # detecta que el prompt pide clasificar una queja
        entrada = texto.split("entrada actual:")[-1]  # extrae el texto de la queja real, despues de la ultima etiqueta
        if "cobr" in entrada or "factura" in entrada:  # busca palabras relacionadas con cobros
            return "facturacion"
        if "senal" in entrada or "lenta" in entrada or "cae" in entrada:  # busca palabras relacionadas con el servicio
            return "calidad_servicio"
        if "llamada" in entrada or "respuesta" in entrada or "espera" in entrada:  # busca palabras relacionadas con atencion
            return "atencion_cliente"
        return "otro"  # si ninguna palabra clave coincide, se clasifica como "otro"

    # --- Simulacion de decision de un agente (secciones 3 y 4) ---
    if "decide la siguiente accion" in texto:
        return contexto.get("accion_sugerida", "finalizar")  # el contexto ya trae precalculada la siguiente accion sugerida

    # --- Simulacion de un agente redactor (seccion 5) ---
    if "redacta un reporte breve" in texto:
        hallazgos = contexto.get("hallazgos", [])  # el contexto trae los hallazgos que el agente investigador encontro
        cuerpo = " ".join(hallazgos)  # concatena los hallazgos en un solo parrafo simple
        return f"Resumen ejecutivo: {cuerpo}"

    # --- Simulacion de un agente revisor (seccion 5) ---
    if "verifica si el reporte es consistente" in texto:
        cifras_citadas = contexto.get("cifras_citadas", [])  # cifras que el reporte menciona
        cifras_confiables = contexto.get("cifras_confiables", [])  # cifras que realmente vienen de una fuente verificada
        inconsistentes = [c for c in cifras_citadas if c not in cifras_confiables]  # cifras citadas que no estan respaldadas
        if inconsistentes:
            return f"rechazado: cifras sin respaldo -> {inconsistentes}"
        return "aprobado"

    # Respuesta por defecto si ningun patron anterior coincide
    return "no tengo suficiente informacion para responder con precision"


## 3. Agente basico: el ciclo observar, decidir, actuar

Como se describe en el texto guia, un agente repite un ciclo simple: observa el estado
de la tarea, decide la siguiente accion, la ejecuta, y vuelve a observar, hasta cumplir
el objetivo o agotar los intentos permitidos. Aqui se implementa ese ciclo de forma
generica, para reutilizarlo despues en los ejemplos de tool calling y multiagentes.

In [3]:
def ejecutar_agente(estado_inicial, funcion_decidir, funcion_ejecutar, funcion_es_meta, maximo_pasos=10):
    """Implementa el ciclo generico observar-decidir-actuar de un agente.

    estado_inicial: diccionario con el estado inicial de la tarea
    funcion_decidir: funcion que, dado el estado actual, devuelve la siguiente accion
    funcion_ejecutar: funcion que, dada una accion y el estado, devuelve el nuevo estado
    funcion_es_meta: funcion que indica si el estado actual cumple el objetivo
    maximo_pasos: limite de seguridad para evitar que el agente quede en un bucle infinito
    """
    estado = estado_inicial  # el estado comienza como el estado inicial recibido
    historial = [("inicio", dict(estado))]  # se guarda un registro de cada paso, para poder inspeccionarlo despues

    for paso in range(maximo_pasos):  # repite el ciclo hasta maximo_pasos veces
        if funcion_es_meta(estado):  # si el estado actual ya cumple el objetivo, se detiene
            break
        accion = funcion_decidir(estado)  # el agente decide la siguiente accion segun el estado actual
        estado = funcion_ejecutar(accion, estado)  # la accion se ejecuta y se obtiene el nuevo estado
        historial.append((accion, dict(estado)))  # se registra la accion tomada y el estado resultante

    return estado, historial  # devuelve el estado final y el historial completo de pasos


## 4. Tool calling: un agente que reprograma citas medicas

Se reproduce el ejemplo del texto guia: un agente que necesita consultar la cita actual
de un paciente, consultar la disponibilidad de horarios, y confirmar el cambio. Cada una
de estas operaciones se implementa como una herramienta (una funcion de Python
independiente), y el agente decide cual invocar en cada paso segun el estado de la
tarea — nunca modifica los datos directamente, solo decide que herramienta usar.

In [4]:
# "Base de datos" en memoria, simulando el sistema de agendamiento de una clinica
citas_pacientes = {
    "paciente_001": {"dia": "jueves", "hora": "10:00"},  # cita actual del paciente simulado
}

horarios_disponibles_proxima_semana = ["lunes 09:00", "martes 14:00", "miercoles 11:00"]  # horarios libres simulados


def herramienta_consultar_cita(id_paciente):
    """Herramienta 1: consulta la cita actual de un paciente en el sistema."""
    return citas_pacientes.get(id_paciente)  # devuelve la cita registrada, o None si no existe


def herramienta_consultar_disponibilidad():
    """Herramienta 2: consulta los horarios disponibles la proxima semana."""
    return list(horarios_disponibles_proxima_semana)  # devuelve una copia de la lista de horarios libres


def herramienta_confirmar_cambio(id_paciente, nuevo_horario):
    """Herramienta 3: confirma el cambio de horario, modificando el sistema de agendamiento.

    Esta es la unica de las tres herramientas que MODIFICA datos, por lo que en un
    sistema real seria la candidata natural a requerir una confirmacion humana antes
    de ejecutarse (ver la seccion 7, sobre riesgos y confirmacion humana).
    """
    dia, hora = nuevo_horario.split(" ")  # separa el texto "lunes 09:00" en dia y hora
    citas_pacientes[id_paciente] = {"dia": dia, "hora": hora}  # sobrescribe la cita del paciente con el nuevo horario
    horarios_disponibles_proxima_semana.remove(nuevo_horario)  # retira ese horario de la lista de disponibles
    return citas_pacientes[id_paciente]  # devuelve la cita ya actualizada, para que el agente la confirme


In [5]:
def decidir_siguiente_herramienta(estado):
    """Logica de decision del agente: que herramienta usar segun lo que ya sabe."""
    if "cita_actual" not in estado:  # si todavia no conoce la cita actual del paciente
        return "consultar_cita"  # primero debe consultarla
    if "horarios_disponibles" not in estado:  # si ya conoce la cita, pero no los horarios disponibles
        return "consultar_disponibilidad"  # debe consultar la disponibilidad
    if "cita_confirmada" not in estado:  # si ya tiene ambos datos, pero aun no confirmo el cambio
        return "confirmar_cambio"  # procede a confirmar el cambio con el primer horario disponible
    return "finalizar"  # si ya confirmo el cambio, no hay nada mas que hacer


def ejecutar_herramienta(accion, estado):
    """Ejecuta la herramienta elegida por el agente y actualiza el estado con el resultado."""
    nuevo_estado = dict(estado)  # se copia el estado para no modificar el original directamente

    if accion == "consultar_cita":
        nuevo_estado["cita_actual"] = herramienta_consultar_cita(estado["id_paciente"])  # guarda la cita consultada
    elif accion == "consultar_disponibilidad":
        nuevo_estado["horarios_disponibles"] = herramienta_consultar_disponibilidad()  # guarda los horarios consultados
    elif accion == "confirmar_cambio":
        horario_elegido = nuevo_estado["horarios_disponibles"][0]  # el agente elige el primer horario disponible
        nuevo_estado["cita_confirmada"] = herramienta_confirmar_cambio(estado["id_paciente"], horario_elegido)  # confirma el cambio

    return nuevo_estado  # devuelve el estado actualizado con el resultado de la herramienta ejecutada


def es_meta_reprogramacion(estado):
    """El objetivo se cumple cuando la cita ya quedo confirmada en el nuevo horario."""
    return "cita_confirmada" in estado  # verifica si la clave de confirmacion ya existe en el estado


In [6]:
estado_inicial = {"id_paciente": "paciente_001"}  # el agente solo conoce el identificador del paciente al empezar

estado_final, historial_pasos = ejecutar_agente(
    estado_inicial=estado_inicial,  # estado con el que arranca el agente
    funcion_decidir=decidir_siguiente_herramienta,  # funcion que elige la siguiente herramienta
    funcion_ejecutar=ejecutar_herramienta,  # funcion que ejecuta la herramienta elegida
    funcion_es_meta=es_meta_reprogramacion,  # funcion que indica cuando terminar
)

for paso, (accion, estado_en_ese_paso) in enumerate(historial_pasos):  # recorre el historial completo paso por paso
    print(f"Paso {paso}: accion = {accion}")  # muestra que accion se tomo en ese paso
    print(f"  Estado resultante: {estado_en_ese_paso}")  # muestra como quedo el estado despues de esa accion

print("\nEstado final:", estado_final)  # muestra el estado final completo, con la cita ya confirmada


Paso 0: accion = inicio
  Estado resultante: {'id_paciente': 'paciente_001'}
Paso 1: accion = consultar_cita
  Estado resultante: {'id_paciente': 'paciente_001', 'cita_actual': {'dia': 'jueves', 'hora': '10:00'}}
Paso 2: accion = consultar_disponibilidad
  Estado resultante: {'id_paciente': 'paciente_001', 'cita_actual': {'dia': 'jueves', 'hora': '10:00'}, 'horarios_disponibles': ['lunes 09:00', 'martes 14:00', 'miercoles 11:00']}
Paso 3: accion = confirmar_cambio
  Estado resultante: {'id_paciente': 'paciente_001', 'cita_actual': {'dia': 'jueves', 'hora': '10:00'}, 'horarios_disponibles': ['lunes 09:00', 'martes 14:00', 'miercoles 11:00'], 'cita_confirmada': {'dia': 'lunes', 'hora': '09:00'}}

Estado final: {'id_paciente': 'paciente_001', 'cita_actual': {'dia': 'jueves', 'hora': '10:00'}, 'horarios_disponibles': ['lunes 09:00', 'martes 14:00', 'miercoles 11:00'], 'cita_confirmada': {'dia': 'lunes', 'hora': '09:00'}}


## 5. Sistema multiagente: un reporte de inteligencia de mercado

Se reproduce el ejemplo del texto guia, dividiendo la tarea entre cuatro agentes
especializados: un investigador que recopila datos, un analista que los organiza, un
redactor que produce el reporte, y un revisor que verifica que las cifras citadas
coincidan con las encontradas por el investigador. Un agente orquestador coordina el
orden en que se ejecutan.

In [7]:
def agente_investigador():
    """Simula la busqueda de informacion sobre un competidor. En un caso real, esta
    funcion haria tool calling contra una API de noticias o una base de datos financiera."""
    return {
        "ingresos_reportados": "crecimiento del 12 por ciento en el ultimo trimestre",
        "cifras_confiables": ["12 por ciento"],  # cifra que SI viene de una fuente verificada por este agente
    }


def agente_analista(datos_investigacion):
    """Organiza los datos crudos del investigador en hallazgos legibles."""
    hallazgos = [
        f"El competidor reporto un {datos_investigacion['ingresos_reportados']}.",  # hallazgo basado en datos reales
        "El competidor reporto un crecimiento del 40 por ciento en nuevos clientes.",  # hallazgo con una cifra NO verificada, a proposito
    ]
    return hallazgos


def agente_redactor(hallazgos):
    """Usa el modelo de lenguaje simulado para redactar un reporte a partir de los hallazgos."""
    prompt = construir_prompt(
        rol="analista de inteligencia de mercado",
        instruccion="redacta un reporte breve con los hallazgos encontrados",
        formato_esperado="un parrafo corto",
        entrada=str(hallazgos),
    )
    return modelo_lenguaje_simulado(prompt, contexto={"hallazgos": hallazgos})  # genera el reporte usando el modelo simulado


def agente_revisor(reporte, cifras_confiables):
    """Verifica que las cifras mencionadas en el reporte tengan respaldo del investigador."""
    cifras_en_el_reporte = [c for c in ["12 por ciento", "40 por ciento"] if c in reporte]  # cifras que aparecen literalmente en el texto
    prompt = construir_prompt(
        rol="revisor de calidad de reportes",
        instruccion="verifica si el reporte es consistente con las cifras confiables",
        formato_esperado="aprobado o rechazado con la lista de cifras sin respaldo",
        entrada=reporte,
    )
    return modelo_lenguaje_simulado(
        prompt,
        contexto={"cifras_citadas": cifras_en_el_reporte, "cifras_confiables": cifras_confiables},
    )


def agente_orquestador():
    """Coordina el orden de ejecucion de los cuatro agentes especializados."""
    datos = agente_investigador()  # paso 1: investigar
    hallazgos = agente_analista(datos)  # paso 2: analizar los datos crudos
    reporte = agente_redactor(hallazgos)  # paso 3: redactar el reporte a partir de los hallazgos
    resultado_revision = agente_revisor(reporte, datos["cifras_confiables"])  # paso 4: revisar el reporte redactado
    return {
        "reporte": reporte,
        "resultado_revision": resultado_revision,
    }


In [8]:
resultado_final = agente_orquestador()  # ejecuta el sistema multiagente completo, en el orden coordinado por el orquestador

print("Reporte redactado:")
print(resultado_final["reporte"])  # muestra el reporte que el agente redactor produjo

print("\nResultado de la revision:")
print(resultado_final["resultado_revision"])  # muestra si el agente revisor aprobo o rechazo el reporte, y por que


Reporte redactado:
Resumen ejecutivo: El competidor reporto un crecimiento del 12 por ciento en el ultimo trimestre. El competidor reporto un crecimiento del 40 por ciento en nuevos clientes.

Resultado de la revision:
rechazado: cifras sin respaldo -> ['40 por ciento']


Nótese que el agente revisor rechaza el reporte: el agente analista incluyó una cifra
("40 por ciento de nuevos clientes") que el investigador nunca verificó. Este es,
precisamente, el tipo de error que un sistema multiagente con un paso de revisión puede
detectar antes de que el reporte llegue a un humano — y también una forma concreta de
mostrar cómo se vería una alucinación dentro de un flujo de varios agentes.

## 6. Simulacion de un flujo tipo n8n

n8n permite construir flujos conectando nodos visualmente, sin escribir codigo de
integracion. Para entender la mecanica que hay detras de esa interfaz visual, aqui se
representa cada nodo como un objeto de Python con una funcion asociada, y se conectan en
una lista, simulando el flujo de cotizaciones descrito en el texto guia: un disparador,
un agente que extrae datos, una consulta a un catalogo de precios, y un envio final.

In [9]:
class NodoFlujo:
    """Representa un nodo de un flujo de automatizacion, similar a un nodo en n8n."""

    def __init__(self, nombre, funcion):
        self.nombre = nombre  # nombre descriptivo del nodo, para mostrarlo en el registro de ejecucion
        self.funcion = funcion  # funcion de Python que el nodo ejecuta cuando le llega el flujo

    def ejecutar(self, datos_entrada):
        """Ejecuta la funcion del nodo sobre los datos que le llegan del nodo anterior."""
        return self.funcion(datos_entrada)  # simplemente delega en la funcion asociada al nodo


catalogo_precios = {"laptop": 2_500_000, "monitor": 650_000, "teclado": 120_000}  # catalogo simulado de precios


def nodo_disparador(correo_entrante):
    """Nodo 1: se activa cuando llega un correo nuevo, y lo deja listo para el siguiente nodo."""
    return {"correo": correo_entrante}  # simplemente empaqueta el correo recibido


def nodo_agente_extraccion(datos):
    """Nodo 2: el agente de IA (aqui, simulado) extrae el producto y la cantidad solicitada."""
    texto = datos["correo"].lower()  # convierte el correo a minusculas para facilitar la busqueda
    productos_encontrados = [p for p in catalogo_precios if p in texto]  # busca que productos del catalogo se mencionan
    datos["productos_solicitados"] = productos_encontrados  # agrega la lista de productos encontrados al flujo
    return datos


def nodo_consultar_catalogo(datos):
    """Nodo 3: consulta el precio de cada producto solicitado en el catalogo."""
    datos["cotizacion"] = {p: catalogo_precios[p] for p in datos["productos_solicitados"]}  # construye la cotizacion producto por producto
    return datos


def nodo_enviar_respuesta(datos):
    """Nodo 4: genera el texto final que se enviaria como respuesta al cliente."""
    lineas = [f"{producto}: ${precio:,}" for producto, precio in datos["cotizacion"].items()]  # formatea cada linea de la cotizacion
    datos["respuesta_generada"] = "Cotizacion:\n" + "\n".join(lineas)  # ensambla el texto final de la respuesta
    return datos


flujo_cotizaciones = [
    NodoFlujo("disparador", nodo_disparador),  # primer nodo del flujo
    NodoFlujo("extraccion", nodo_agente_extraccion),  # segundo nodo: extraccion con el agente
    NodoFlujo("catalogo", nodo_consultar_catalogo),  # tercer nodo: consulta de precios
    NodoFlujo("respuesta", nodo_enviar_respuesta),  # cuarto nodo: generacion de la respuesta final
]


def ejecutar_flujo(flujo, dato_inicial):
    """Ejecuta una lista de nodos en orden, pasando el resultado de uno como entrada del siguiente."""
    resultado = dato_inicial  # el primer nodo recibe el dato inicial
    for nodo in flujo:  # recorre cada nodo del flujo en el orden en que fueron definidos
        print(f"Ejecutando nodo: {nodo.nombre}")  # muestra en que nodo esta el flujo en este momento
        resultado = nodo.ejecutar(resultado)  # ejecuta el nodo actual y guarda su resultado
    return resultado  # devuelve el resultado final, despues de pasar por todos los nodos


In [10]:
correo_simulado = "Hola, quisiera cotizar un laptop y un monitor para mi oficina, gracias."  # correo de ejemplo de un cliente

resultado_flujo = ejecutar_flujo(flujo_cotizaciones, correo_simulado)  # ejecuta el flujo completo sobre ese correo

print("\n" + resultado_flujo["respuesta_generada"])  # muestra la respuesta final generada por el flujo


Ejecutando nodo: disparador
Ejecutando nodo: extraccion
Ejecutando nodo: catalogo
Ejecutando nodo: respuesta

Cotizacion:
laptop: $2,500,000
monitor: $650,000


## 7. Riesgos en la practica: alucinaciones y confirmacion humana

Dos de los riesgos mencionados en el texto guia se pueden ilustrar directamente con el
codigo ya construido: que pasa cuando el modelo no tiene suficiente informacion para
responder con certeza, y como se protege una accion irreversible con un paso de
confirmacion humana.

In [11]:
# Alucinacion: se le pide al modelo simulado algo que ningun patron conocido cubre
prompt_fuera_de_alcance = construir_prompt(
    rol="asistente general",
    instruccion="dime la cifra exacta de empleados que tendra la empresa en 5 anios",
    formato_esperado="un numero",
    entrada="proyeccion de personal",
)

respuesta = modelo_lenguaje_simulado(prompt_fuera_de_alcance)  # el modelo simulado no tiene ningun patron para esto
print("Respuesta del modelo ante una pregunta sin informacion real:", respuesta)

print(
    "\nEn un LLM real, en lugar de admitir que no sabe, el modelo podria inventar un "
    "numero que suena razonable pero no tiene ningun respaldo real: eso es una "
    "alucinacion. La defensa contra esto no es confiar en que el modelo 'sepa cuando no "
    "sabe', sino obligarlo a basarse en datos obtenidos por tool calling (como en la "
    "seccion 5), y rechazar o marcar como no verificada cualquier cifra que no venga de "
    "una fuente confirmada."
)


Respuesta del modelo ante una pregunta sin informacion real: no tengo suficiente informacion para responder con precision

En un LLM real, en lugar de admitir que no sabe, el modelo podria inventar un numero que suena razonable pero no tiene ningun respaldo real: eso es una alucinacion. La defensa contra esto no es confiar en que el modelo 'sepa cuando no sabe', sino obligarlo a basarse en datos obtenidos por tool calling (como en la seccion 5), y rechazar o marcar como no verificada cualquier cifra que no venga de una fuente confirmada.


In [12]:
def herramienta_confirmar_cambio_con_aprobacion_humana(id_paciente, nuevo_horario, aprobado_por_humano):
    """Version mas segura de la herramienta de confirmacion, usada en la seccion 4.

    A diferencia de la version original, esta exige una aprobacion humana explicita
    antes de modificar datos reales, como medida de control sobre una accion
    irreversible.
    """
    if not aprobado_por_humano:  # si nadie aprobo explicitamente el cambio
        return "accion bloqueada: se requiere aprobacion humana antes de confirmar el cambio de cita"

    return herramienta_confirmar_cambio(id_paciente, nuevo_horario)  # si fue aprobado, se ejecuta la accion real


# Intento sin aprobacion humana: la accion queda bloqueada
print(herramienta_confirmar_cambio_con_aprobacion_humana("paciente_001", "martes 14:00", aprobado_por_humano=False))

# Intento con aprobacion humana explicita: la accion se ejecuta normalmente
print(herramienta_confirmar_cambio_con_aprobacion_humana("paciente_001", "martes 14:00", aprobado_por_humano=True))


accion bloqueada: se requiere aprobacion humana antes de confirmar el cambio de cita
{'dia': 'martes', 'hora': '14:00'}


## Cierre

Este notebook mostro, de forma practica y comentada linea a linea, la mecanica interna
de los temas de la Unidad 2.3: como se estructura un prompt, como funciona el ciclo de
decision de un agente, como el tool calling separa la decision (del modelo) de la
ejecucion (del sistema), como se coordina un sistema multiagente con un paso de
revision, como se representa un flujo tipo n8n como una secuencia de nodos, y como se
protege una accion irreversible con confirmacion humana.

Para el proyecto final, el ejercicio equivalente es tomar el flujo de trabajo repetitivo
identificado para la organizacion elegida, y disenar (en codigo, en n8n, o solo en
papel) que herramientas necesitaria el agente y en que paso haria falta una
confirmacion humana antes de actuar.